
Salaries Differences 
Calculate the difference between the highest salaries in the marketing and engineering departments. Output just the absolute difference in salaries. Tables db_employee and db_dept

In [0]:
data_emp = [
    (1, "John", "Doe", 60000, 1),
    (2, "Jane", "Smith", 80000, 2),
    (3, "Sam", "Brown", 75000, 1),
    (4, "Lisa", "Ray", 90000, 2),
    (5, "Tom", "Hanks", 50000, 1)
]

data_dept = [
    (1, "marketing"),
    (2, "engineering")
]

In [0]:
from pyspark.sql import session
from pyspark.sql.functions import col, max, abs
spark = session.SparkSession.builder.appName("Salary differene").getOrCreate();

In [0]:
#define column names
columns_emp = ["emp_id", "emp_first_name", "emp_last_name", "salary", "dept_id"]
columns_dept = ["dept_id", "dept_name"]

#create dataframes
df_emp = spark.createDataFrame(data_emp, columns_emp)
df_dept = spark.createDataFrame(data_dept, columns_dept)

#display dataframes
display(df_emp)
display(df_dept)

emp_id,emp_first_name,emp_last_name,salary,dept_id
1,John,Doe,60000,1
2,Jane,Smith,80000,2
3,Sam,Brown,75000,1
4,Lisa,Ray,90000,2
5,Tom,Hanks,50000,1


dept_id,dept_name
1,marketing
2,engineering


In [0]:
emp_with_dept_df = df_emp.join(df_dept, on="dept_id",how ="inner")
emp_with_dept_df.show()

+-------+------+--------------+-------------+------+-----------+
|dept_id|emp_id|emp_first_name|emp_last_name|salary|  dept_name|
+-------+------+--------------+-------------+------+-----------+
|      1|     1|          John|          Doe| 60000|  marketing|
|      2|     2|          Jane|        Smith| 80000|engineering|
|      1|     3|           Sam|        Brown| 75000|  marketing|
|      2|     4|          Lisa|          Ray| 90000|engineering|
|      1|     5|           Tom|        Hanks| 50000|  marketing|
+-------+------+--------------+-------------+------+-----------+



In [0]:

result_df = emp_with_dept_df \
    .groupBy() \
    .pivot("dept_name") \
    .agg(max("salary")) \
    .withColumn("salary_difference", abs(col("engineering") - col("marketing")))

result_df.show()

+-----------+---------+-----------------+
|engineering|marketing|salary_difference|
+-----------+---------+-----------------+
|      90000|    75000|            15000|
+-----------+---------+-----------------+



In [0]:


# 1. Group by nothing to aggregate globally, pivot the department names, and get max salary
df_pivot = emp_with_dept_df.groupBy().pivot("dept_name").agg(max("salary"))
df_pivot.show()

# 2. Safely calculate the absolute difference between the newly created columns
df_result = df_pivot.withColumn("salary_difference", abs(col("engineering") - col("marketing")))

# Display the final matrix
df_result.show()

+-----------+---------+
|engineering|marketing|
+-----------+---------+
|      90000|    75000|
+-----------+---------+

+-----------+---------+-----------------+
|engineering|marketing|salary_difference|
+-----------+---------+-----------------+
|      90000|    75000|            15000|
+-----------+---------+-----------------+



In [0]:


# ==============================================================================
# STEP 1: PIVOTING THE DATA
# We are taking the 'long' table (many rows) and turning it into a 'wide' table 
# (side-by-side columns) so we can compare departments directly.
# ==============================================================================

# groupBy(): We group globally (no specific column) to treat the whole table as one unit.
# pivot("dept_name"): We take the values in 'dept_name' and turn them into new column headers.
# agg(max("salary")): For every 'dept_name', find the highest salary and put it under its column.
df_pivot = emp_with_dept_df \
    .groupBy() \
    .pivot("dept_name") \
    .agg(max("salary"))

# ==============================================================================
# STEP 2: CALCULATING THE DIFFERENCE
# Now that 'marketing' and 'engineering' are side-by-side, we perform the math.
# ==============================================================================

# withColumn(): Creates the new 'salary_difference' column.
# abs(col("engineering") - col("marketing")): Subtracts the two columns and 
# ensures the result is always positive using the absolute value function.
df_result = df_pivot.withColumn(
    "salary_difference", 
    abs(col("engineering") - col("marketing"))
)

# Display the final, clean result
df_result.show()

+-----------+---------+-----------------+
|engineering|marketing|salary_difference|
+-----------+---------+-----------------+
|      90000|    75000|            15000|
+-----------+---------+-----------------+

